In [14]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import numpy as np
import pandas as pd
import pickle

In [7]:
# Load the trained model
model=load_model('model.h5')
 
## load the encoder and scaler 
with open('label_encoder_gender.pkl', 'rb') as f:
    label_encoder_gender = pickle.load(f)

with open('scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

with open('onehot_encoder.pkl', 'rb') as f:
    label_encoder = pickle.load(f)

In [8]:
input_data ={
    'CreditScore': 750,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 35,
    'Tenure': 5,
    'Balance': 100000.0,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000.0,
}

In [9]:
import os

print(os.getcwd())
print(os.listdir())

/Users/harpreetsingh/Documents/Churn_Project
['scaler.pkl', 'prediction.ipynb', 'Churn_Modelling.csv', 'experiments.ipynb', 'label_encoder_gender.pkl', 'logs', 'onehot_encoder.pkl', 'model.h5']


In [10]:
# Example nput Data
input_data={
    'CreditScore':600,
    'Geography':'France',
    'Gender':'Male',
    'Age':40,
    'Tenure':3,
    'Balance':60000,
    'NumOfProducts':2,
    'HasCrCard':1,
    'IsActiveMember':1,
    'EstimatedSalary':50000
}

In [23]:
#one-hot encode the categorical features
geo_encoded=label_encoder.transform([[input_data['Geography']]]).toarray()
geo_encoded_df=pd.DataFrame(geo_encoded,columns=label_encoder.get_feature_names_out(['Geography']))
geo_encoded_df

/Users/harpreetsingh/Documents/python_project/venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [15]:
input_df=pd.DataFrame([input_data])
input_df


,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


In [22]:
input_df['Gender'] = label_encoder_gender.transform('Male' if input_df['Gender'][0] == 'Male' else 'Female')

ValueError: y should be a 1d array, got an array of shape () instead.

In [25]:
input_df=pd.concat([input_df.drop("Geography",axis=1),geo_encoded_df],axis=1)
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [26]:
input_scaled = scaler.transform(input_df)

In [27]:
input_scaled

array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

In [28]:
prediction = model.predict(input_scaled)
prediction

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step


array([[0.05599744]], dtype=float32)

In [29]:
prediction_probability = prediction[0][0]
if prediction_probability > 0.5:
    print(f"The customer is likely to churn with a probability of {prediction_probability:.2f}")
else:
    print(f"The customer is unlikely to churn with a probability of {prediction_probability:.2f}")

The customer is unlikely to churn with a probability of 0.06
